In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os

import numpy  as np
import pandas as pd
import h5py
from pathlib import Path

from astropy    import units as u
import matplotlib
from matplotlib import pyplot as plt
import matplotlib.patches as mpatches
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'sans-serif'

font = {'family' : 'sans-serif',
        'size'   : 8}

matplotlib.rc('font', **font)

from scipy.interpolate import RegularGridInterpolator
from scipy.signal      import argrelextrema

EHT = os.path.abspath("../2017_sgra_paper5")
RAPTOR = os.path.abspath("../RAPTOR")
KNS = os.path.abspath("../mod")

sys.path.append(EHT)
    
from common import hallmark as hm
from common import mockservation as mk
from common import io_raptor as io
from common import dalt
from common import viz

sys.path.remove(EHT)

sys.path.append(RAPTOR)
from python.plotting import rapplot
sys.path.remove(RAPTOR)

sys.path.append(KNS)
import module
import plotscripts 
from plotscripts import subtract, image2d, loadfigure, getimagedata


In [2]:
#Computing relevant constants
M=4.14e6  * rapplot.MSUN
d= 8.127* rapplot.KPC

rg = (rapplot.G*M/rapplot.SPEED_OF_LIGHT**2.)

mas = (rg/d)* rapplot.MAS_IN_DEG
print(mas*1000)

Tunit =rg/rapplot.SPEED_OF_LIGHT

halfrange=20 #in rg

5.028714974850079


In [3]:
def import_ab(a, i, cond):
    file = f'../cache/shadows/shadow_a{a:.2f}_i{i:g}.h5' # change this to your local folder
    with h5py.File(file) as h:
      alpha = h['a'][:]
      beta = h['b'][:]

    if cond:
      index_apos = np.where(alpha > 0)[0]
      aright = index_apos[beta[index_apos].argmin()]

      for bb in np.linspace(-10e-3, 10e-3, 10):
        alpha = np.append(alpha, alpha[aright])
        beta = np.append(beta, bb)

    return (alpha, beta)

## Plot High Res image

In [4]:
plt.figure(figsize=(30,30),facecolor='w')
fig, ax = plt.subplots(1,1,figsize=(5,4), sharex=True, sharey=True)
img_path = "../cache/Ma101_w1/Rh1_i60"

# get image
data_id_all, min, max, image_array = subtract(img_path+"/n100", img_path+"/n0", 999 , ave=False)
# data_id_all, min, max, image_array = getimagedata(img_path+'/n100', 999)


#get critical curve 
alpha1, beta1 = import_ab(1.01, 60, True) 

# plot original arc 
ax.plot(alpha1, beta1, linewidth=1, color='deepskyblue', linestyle='-')
# plot connecting line 
# ax.plot(alpha_closed, beta_closed, linewidth=1, color='deepskyblue', linestyle='--')

figure = loadfigure(image_array, np.min(image_array), np.max(image_array), fig, ax, 20, 1, label="Stokes", cmap="afmhot")

ax.set_xlabel(r"x [mas]")
ax.set_ylabel(r"y [mas]")
ax.set_xlim(-8, 8)
ax.set_ylim(-8, 8)
fig.savefig(f"../plot/HPR.pdf", dpi=1000)

read from ../cache/Ma101_w1/Rh1_i60/n100 and ../cache/Ma101_w1/Rh1_i60/n0
Reading keys from:  ../cache/Ma101_w1/Rh1_i60/n100/img_data_999.h5
['I2.300000e+11', 'Q2.300000e+11', 'U2.300000e+11', 'V2.300000e+11', 'alpha', 'beta', 'tau2.300000e+11', 'tauF2.300000e+11']
Reading keys from:  ../cache/Ma101_w1/Rh1_i60/n0/img_data_999.h5
['I2.300000e+11', 'Q2.300000e+11', 'U2.300000e+11', 'V2.300000e+11', 'alpha', 'beta', 'tau2.300000e+11', 'tauF2.300000e+11']
Reading in:  ../cache/Ma101_w1/Rh1_i60/n100/img_data_999.h5
<KeysViewHDF5 ['I2.300000e+11', 'Q2.300000e+11', 'U2.300000e+11', 'V2.300000e+11', 'alpha', 'beta', 'tau2.300000e+11', 'tauF2.300000e+11']>
8
Reading in:  ../cache/Ma101_w1/Rh1_i60/n0/img_data_999.h5
<KeysViewHDF5 ['I2.300000e+11', 'Q2.300000e+11', 'U2.300000e+11', 'V2.300000e+11', 'alpha', 'beta', 'tau2.300000e+11', 'tauF2.300000e+11']>
8
Reading keys from:  ../cache/Ma101_w1/Rh1_i60/n100/img_data_1000.h5
['I2.300000e+11', 'Q2.300000e+11', 'U2.300000e+11', 'V2.300000e+11', 'alph

In [5]:
plt.figure(figsize=(30,30),facecolor='w')
fig, ax = plt.subplots(1,1,figsize=(5,4), sharex=True, sharey=True)
figure = loadfigure(image_array, np.min(image_array), np.max(image_array), fig, ax, 20, 1, label="Stokes", cmap="afmhot")
ax.set_xlabel(r"x [mas]")
ax.set_ylabel(r"y [mas]")
ax.set_xlim(-3, 1)
ax.set_ylim(-4, 4)

fig.colorbar(figure, ax=ax)
fig.savefig(f"../plot/HPRi60_zoom.pdf", dpi=1000)

## Higher order photon rings structures

In [6]:
# Load all images 
pf = hm.ParaFrame('../cache/{mag}a{aspin:g}_w{window:d}/Rh{Rhigh:d}_i{inc:d}/n{n:n}/img_data_{snapshot:d}.h5')
sel = pf(snapshot=[999, 1000, 101000])
display(sel)

/Users/suzukihina/Documents/UR/NkS/KNS_PR/2017_sgra_paper5/common/hallmark.py:26: FutureWarning: Logical ops (and, or, xor) between Pandas objects and dtype-less sequences (e.g. list, tuple) are deprecated and will raise in a future version. Wrap the object in a Series, Index, or np.array before operating instead.
  mask |= self[k].isin(v)


,path,mag,aspin,window,Rhigh,inc,n,snapshot
0,../cache/Ma101_w1/Rh1_i15/n0/img_data_999.h5,M,101.0,1,1,15,0,999
1,../cache/Ma101_w1/Rh1_i15/n1/img_data_1000.h5,M,101.0,1,1,15,1,1000
2,../cache/Ma101_w1/Rh1_i15/n1/img_data_101000.h5,M,101.0,1,1,15,1,101000
3,../cache/Ma101_w1/Rh1_i15/n1/img_data_999.h5,M,101.0,1,1,15,1,999
4,../cache/Ma101_w1/Rh1_i15/n100/img_data_1000.h5,M,101.0,1,1,15,100,1000
5,../cache/Ma101_w1/Rh1_i15/n100/img_data_999.h5,M,101.0,1,1,15,100,999
6,../cache/Ma101_w1/Rh1_i15/n2/img_data_1000.h5,M,101.0,1,1,15,2,1000
7,../cache/Ma101_w1/Rh1_i15/n2/img_data_101000.h5,M,101.0,1,1,15,2,101000
8,../cache/Ma101_w1/Rh1_i15/n2/img_data_999.h5,M,101.0,1,1,15,2,999
9,../cache/Ma101_w1/Rh1_i30/n0/img_data_999.h5,M,101.0,1,1,30,0,999


In [7]:
ns = np.sort( np.array(sel.n.unique()))
incs = np.sort(np.array(sel.inc.unique())) 

print("n: ", ns)
print("inc: ", incs[0:len(incs)])

n:  [  0   1   2 100]
inc:  [15 30 45 60 75]


In [8]:
# substruct 
# for n in [1,2]: 
#     for inc in incs: 

#         try:
#             s_0 = sel[(sel['inc']==inc) & (sel['n']==n ) & (sel['snapshot']==999)]
#             s_1 = sel[(sel['inc']==inc) & (sel['n']==(n-1)) & (sel['snapshot']==999) ]

#             path_object_0 = Path(s_0.path.item())
#             img_path_0 = str( path_object_0.parent )
#             path_object_1 = Path(s_1.path.item())
#             img_path_1 = str( path_object_1.parent )

#             subtract(img_path_0, img_path_1, 999 , ave=False)

#         except: 
#             print("Error: ", inc, n)


In [9]:
fig, axes = plt.subplots(len(incs), len(ns) + 1, figsize=(len(ns) + 1, len(incs)), sharex=True, sharey=True)

for row, inc in enumerate(incs):  # row = inclination
    imgs_list = []
    alpha1, beta1 = import_ab(1.01, inc, True) 
    
    for col, n in enumerate(ns):  # col = photon ring order
        ax = axes[row, col]

        # images ----------------------------------------------------------------------
        if n in [1, 2]:
            s = sel[(sel['inc'] == inc) & (sel['n'] == n) & (sel['snapshot'] == 101000)]
            snapshot = 101000
        else:
            s = sel[(sel['inc'] == inc) & (sel['n'] == n) & (sel['snapshot'] == 999)]
            snapshot = 999

        if s.path.shape == (1,):
            path_object = Path(s.path.item())
            img_path = str(path_object.parent)
            if snapshot==101000:
                data_id_all, min, max, image_array = getimagedata(img_path, snapshot, resize=False)
            else: 
                data_id_all, min, max, image_array = getimagedata(img_path, snapshot, resize=True)
                
            figure = loadfigure(image_array, min, max, fig, ax, halfrange=20, mas=1, label="Stokes", cmap="afmhot")
            if n==100: 
                ax.plot(alpha1, beta1, linewidth=1, color='deepskyblue', linestyle='-')  
            ax.set_xlim(-8, 8)
            ax.set_ylim(-8, 8)
            ax.set_aspect('equal')
        else:
            print("error", inc, n)
            continue
        ax.tick_params(axis='x', direction='in')
        ax.tick_params(axis='y', direction='in')
        if n in [0, 1, 2]:
            imgs_list.append(image_array)

        if row == 0:
            if n == 100:
                ax.set_title("Total", fontsize=10)
            else:
                ax.set_title(f"n = {n}", fontsize=10)
                
        
        # region plot ------------------------------------------------
        imgs = np.stack(imgs_list, axis=0)
        legend_handles = module.plotregion(imgs, axes[row, -1])
        axes[row, -1].tick_params(direction='in')
        
        axes[row, col].set_aspect('equal')
        axes[row, col].tick_params(direction='in')

        if col ==0:
            ax.set_ylabel(f"i = {inc}°", fontsize=10, loc='bottom')


# Global legend
fig.legend(handles=legend_handles, bbox_to_anchor=(1.1, 0.9), frameon=False)

# Add colorbar
cbar_ax = fig.add_axes([0.9, 0.1, 0.02, 0.6])
fig.colorbar(figure, cax=cbar_ax, orientation='vertical')

fig.text(0.5, 0.04, r"x [$r_g$]", ha='center', fontsize=10)  
fig.text(-0.14, 0.5, r"y [$r_g$]", va='center', rotation='vertical', fontsize=10)  

fig.subplots_adjust(left=0, right=0.85, top=0.9, bottom=0.1, wspace=0.1, hspace=0.1)

# Save
fig.savefig("../plot/hpr_withcurve.jpg", bbox_inches='tight', dpi=300)

Reading keys from:  ../cache/Ma101_w1/Rh1_i15/n0/img_data_999.h5
['I2.300000e+11', 'Q2.300000e+11', 'U2.300000e+11', 'V2.300000e+11', 'alpha', 'beta', 'tau2.300000e+11', 'tauF2.300000e+11']
Reading in:  ../cache/Ma101_w1/Rh1_i15/n0/img_data_999.h5
<KeysViewHDF5 ['I2.300000e+11', 'Q2.300000e+11', 'U2.300000e+11', 'V2.300000e+11', 'alpha', 'beta', 'tau2.300000e+11', 'tauF2.300000e+11']>
8
Reading keys from:  ../cache/Ma101_w1/Rh1_i15/n1/img_data_101000.h5
['I2.300000e+11', 'Q2.300000e+11', 'U2.300000e+11', 'V2.300000e+11', 'alpha', 'beta', 'tau2.300000e+11', 'tauF2.300000e+11']
Reading in:  ../cache/Ma101_w1/Rh1_i15/n1/img_data_101000.h5
<KeysViewHDF5 ['I2.300000e+11', 'Q2.300000e+11', 'U2.300000e+11', 'V2.300000e+11', 'alpha', 'beta', 'tau2.300000e+11', 'tauF2.300000e+11']>
8
file converted
Reading keys from:  ../cache/Ma101_w1/Rh1_i15/n2/img_data_101000.h5
['I2.300000e+11', 'Q2.300000e+11', 'U2.300000e+11', 'V2.300000e+11', 'alpha', 'beta', 'tau2.300000e+11', 'tauF2.300000e+11']
Readin

## VAs per angle

In [10]:
ns = [100,0, 1,2]
i=45

In [11]:
fig, axes = plt.subplots(3, 2, figsize=(8, 7), sharey=True, sharex=True)

legend_handles = []  
axes = axes.flatten()  

for row, inc in enumerate(incs):
    ax = axes[row]
    
    for col, n in enumerate(ns): 
        if n in [1, 2]: 
            s = sel[(sel['inc'] == inc) & (sel['n'] == n) & (sel['snapshot'] == 1000)]
            path_object = Path(s.path.item())
            mov = io.load_mov(s.path, [1000], 20, resize = True)
        else: 
            s = sel[(sel['inc'] == inc) & (sel['n'] == n) & (sel['snapshot'] == 999)]
            path_object = Path(s.path.item())
            mov = io.load_mov(s.path, [999], 20, resize = True)
        
        # Plot VA
        handles = module.plot_va(mov, ax, n, i)
        if handles:
            legend_handles.extend(handles)

    # add lines for EHT, BHEX 
    ax.axvline(x=8.6, color='black', linestyle='--', lw=0.8)
    ax.axvline(x=35, color='black', linestyle='--', lw=0.8)
    
    if inc == 75: 
        ax.text(8.6, 0, "EHT", transform=ax.get_xaxis_transform(), rotation=-45)
        ax.text(35, 0, "BHEX", transform=ax.get_xaxis_transform(), rotation=-45)
    

    # layout
    ax.text(0.95, 0.9, f"i = {inc}°", transform=ax.transAxes,
            fontsize=10, ha='right', va='top')
    ax.set_xlim(0,100)
    ax.set_ylim(1e-5,1)
    print("DONE:", inc)


fig.subplots_adjust(wspace=0.05, hspace=0.1)

# change the last panel to space for legend
axes[-1].axis('off')
legend_ax = axes[-1]
legend_ax.legend(
    ncol=2,
    handles=legend_handles[:4],
    fontsize=10,
    loc="center",
    frameon=True
)

fig.supxlabel(r"Baseline Length u ($G\lambda$)", fontsize=10)
fig.supylabel("Visibility Amplitude (Jy)", fontsize=10)   
fig.savefig(f"../plot/visibility.pdf")

DONE: 15
DONE: 30
DONE: 45
DONE: 60
DONE: 75
